# Loss of Exclusivity Revenue Forecaster

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Build time-series forecasts** using `SNOWFLAKE.ML.FORECAST`
2. **Predict 24-month revenue impact** of patent expiries
3. **Model LOE scenarios** with exponential decline curves
4. **Calculate confidence intervals** for best/worst case
5. **Generar portfolio risk** analysis across products

---

## What You'll Build

A **revenue forecasting system** that predicts patent expiry impact:
- 24-month revenue forecasts using ML
- LOE impact modeling (80-90% revenue decline)
- Confidence intervals (optimistic/pessimistic scenarios)
- Portfolio risk analysis (revenue at risk by product)
- Strategic planning dashboard

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 15 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features Used

- `SNOWFLAKE.ML.FORECAST` - Time-series ML for revenue prediction
- `!FORECAST(FORECASTING_PERIODS => 24)` - Generar 24-month forecast
- `POWER()` - Model exponential revenue decline curves
- `DATEADD()` - Create future time periods
- `UNIFORM()` - Generar realistic revenue patterns

Let's begin!

---

## Paso 1: Configuración del Entorno

### Loss of Exclusivity (LOE)

Patent expiry → generic competition → 80-90% revenue decline over 12-24 months

In [ ]:
CREATE DATABASE IF NOT EXISTS LOE_FORECAST_DB;
USE SCHEMA LOE_FORECAST_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL';
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db, CURRENT_SCHEMA() as schema;

---

## Paso 2: Create Tables

In [ ]:
CREATE OR REPLACE TABLE product_revenue_history (
    revenue_date DATE,
    product_name VARCHAR(100),
    monthly_revenue_usd DECIMAL(15,2)
);

CREATE OR REPLACE TABLE loe_events (
    product_name VARCHAR(100) PRIMARY KEY,
    patent_expiry_date DATE,
    expected_generic_entry_date DATE
);

---

## Paso 3: Generar 36 Months Historical Data

Realistic growth pre-LOE, decline post-LOE

In [ ]:
-- Insert LOE events
INSERT INTO loe_events VALUES
    ('Xarelto', '2024-07-31', '2024-10-31'),
    ('Eylea', '2025-11-30', '2026-02-28'),
    ('Stivarga', '2027-09-30', NULL);

-- Generar 36 months revenue with LOE impact
INSERT INTO product_revenue_history
WITH months AS (
    SELECT DATEADD('month', -SEQ4(), CURRENT_DATE()) as rev_date
    FROM TABLE(GENERATOR(ROWCOUNT => 36))
),
revenue_calc AS (
    SELECT 
        m.rev_date,
        le.product_name,
        CASE le.product_name
            WHEN 'Xarelto' THEN 2400000000
            WHEN 'Eylea' THEN 2100000000
            ELSE 450000000
        END as base_revenue,
        -- Apply decline post-LOE
        CASE 
            WHEN m.rev_date < le.patent_expiry_date 
                THEN base_revenue * (1 + ROW_NUMBER() OVER (PARTITION BY le.product_name ORDER BY m.rev_date) * 0.005)
            ELSE base_revenue * POWER(0.85, DATEDIFF('month', le.patent_expiry_date, m.rev_date))
        END as monthly_rev
    FROM months m
    CROSS JOIN loe_events le
)
SELECT rev_date, product_name, ROUND(monthly_rev * (1 + UNIFORM(-0.05,0.05,RANDOM())), 2)
FROM revenue_calc
ORDER BY product_name, rev_date;

SELECT product_name, COUNT(*) as months, ROUND(AVG(monthly_revenue_usd)/1000000,1) as avg_rev_millions
FROM product_revenue_history GROUP BY product_name;

---

## Paso 4: Train FORECAST Models

### Qué Vamos a Crear

**Time-series forecasting models** that predict future revenue patterns based on historical data, enabling proactive LOE planning.

### Understanding `SNOWFLAKE.ML.FORECAST`

Snowflake's **native time-series forecasting function** powered by machine learning:

```sql
CREATE SNOWFLAKE.ML.FORECAST model_name(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'training_data'),
    TIMESTAMP_COLNAME => 'date_column',
    TARGET_COLNAME => 'value_to_forecast'
);
```

**Returns**: A trained ML model object with methods for forecasting

### Why ML.FORECAST for Revenue Prediction?

`SNOWFLAKE.ML.FORECAST` is ideal for LOE (Loss of Exclusivity) forecasting because:

- **Automatic Algorithm Selection**: Tests ARIMA, Prophet, and other time-series algorithms
- **No Python Required**: 100% SQL - accessible to business analysts
- **Seasonality Detection**: Automatically identifies yearly, quarterly, monthly patterns
- **Trend Analysis**: Captures growth/decline trajectories
- **Confidence Intervals**: Provides best/worst case scenarios
- **Scalable**: Train separate models for hundreds of products simultaneously

### How It Works (Behind the Scenes)

When you call `CREATE SNOWFLAKE.ML.FORECAST`, Snowflake:

1. **Analyzes Time-Series Data**: Examines historical patterns, seasonality, trends
2. **Detects Frequency**: Auto-identifies if data is daily, monthly, quarterly
3. **Tests Multiple Algorithms**: 
   - ARIMA (AutoRegressive Integrated Moving Average)
   - Prophet (Facebook's time-series algorithm)
   - ETS (Exponential Smoothing)
4. **Selects Best Model**: Chooses algorithm with lowest forecast error
5. **Trains on Full Dataset**: Optimizes parameters for your specific data
6. **Creates Prediction Function**: Returns model with `!FORECAST()` method

**Training Time**: 30-120 seconds depending on data volume and complexity

### Real-World LOE Example

**Historical Revenue Data** (Xarelto - 36 months):
```
Month        | Revenue (M)
-------------|------------
2021-01      | $85.2
2021-02      | $88.7
2021-03      | $91.3
...          | ...
2024-01      | $245.8  ← Last observed
```

**Training Process**:
- Model learns: "Revenue growing ~8% quarterly with Q4 seasonality spike"
- Discovers patterns: Q1 dip (insurance resets), Q4 surge (year-end prescribing)

**Result**: Model can now project revenue for months 37-60 (next 2 years)!

### Key Parameters Explained

#### **1. `INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'view_name')`**

Passes a reference to your training data table or view:

```sql
INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'xarelto_training_data')
-- or
INPUT_DATA => SYSTEM$QUERY_REFERENCE('SELECT date, revenue FROM history WHERE product = ''Xarelto''')
```

**What It Does**:
- `SYSTEM$REFERENCE()` creates a **privileged reference** to data
- Model inherits your data access permissions
- No data copying - references original table

**Critical Requirements**:
- Table/view must have at least 2 columns: timestamp + target value
- Minimum 10 data points (preferably 24+ for monthly data)
- No gaps in time series (consecutive periods)

#### **2. `TIMESTAMP_COLNAME => 'column_name'`**

Specifies which column contains the time dimension:

```sql
TIMESTAMP_COLNAME => 'revenue_date'
```

**What It Does**:
- Identifies the temporal ordering of data points
- Used to detect frequency (daily, monthly, quarterly)
- Must be DATE, TIMESTAMP, or TIMESTAMP_NTZ type

**Supported Frequencies**:
- Daily (at least 100 data points recommended)
- Weekly (at least 52 points recommended)
- Monthly (at least 24 points recommended)
- Quarterly (at least 12 points recommended)
- Yearly (at least 5 points recommended)

#### **3. `TARGET_COLNAME => 'column_name'`**

Specifies what value you want to forecast:

```sql
TARGET_COLNAME => 'monthly_revenue_usd'
```

**What It Does**:
- The metric model will predict
- Must be numeric (INTEGER, FLOAT, DECIMAL)
- Typically: revenue, units sold, prescriptions, market share

**Best Practices**:
- Use **stable metrics** (avoid highly volatile data)
- Consider **log transformation** for exponential growth
- **Aggregate to appropriate frequency** (monthly better than daily for revenue)

### Two-Paso Training Process

**Paso 1: Create Training Views**

Filter historical data to only relevant period/product:

```sql
CREATE OR REPLACE VIEW xarelto_training_data AS
SELECT 
    revenue_date,           -- TIMESTAMP column
    monthly_revenue_usd     -- TARGET column
FROM product_revenue_history
WHERE product_name = 'Xarelto'
  AND revenue_date >= '2021-01-01'   -- Last 3 years of data
ORDER BY revenue_date;                -- Chronological order
```

**Why Views?**:
- Clean, filtered data for each product
- Easy to retrain with updated data
- Reusable for multiple forecasting scenarios

**Paso 2: Train Forecast Models**

```sql
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST xarelto_forecast(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'xarelto_training_data'),
    TIMESTAMP_COLNAME => 'revenue_date',
    TARGET_COLNAME => 'monthly_revenue_usd'
);
```

**What Gets Created**: A **model object** with methods:
- `!FORECAST(FORECASTING_PERIODS => n)` - Generar predictions
- `!SHOW_EVALUATION_METRICS()` - View accuracy metrics
- `!SHOW_TRAINING_LOGS()` - Debug model training
- `!EXPLAIN()` - Understand model behavior

### Advanced Configuration (Optional)

You can fine-tune model training with `CONFIG_OBJECT`:

```sql
CREATE SNOWFLAKE.ML.FORECAST model_name(
    INPUT_DATA => ...,
    TIMESTAMP_COLNAME => ...,
    TARGET_COLNAME => ...,
    CONFIG_OBJECT => {
        'frequency': 'M',              -- Force monthly (M), quarterly (Q), daily (D)
        'aggregation_frequency': 'M',  -- Aggregate data to monthly
        'lower_bound': 0,              -- Revenue can't be negative
        'upper_bound': 500000000,      -- Max realistic revenue ($500M)
        'method': 'ARIMA'              -- Force specific algorithm (ARIMA, PROPHET, ETS)
    }
);
```

**When to Use CONFIG_OBJECT**:
- **Frequency mismatch**: Data is daily but you want monthly forecasts
- **Known constraints**: Revenue can't be negative
- **Algorithm preference**: Force ARIMA for business reporting consistency

### Real-World Training Example

```sql
-- Product: Eylea (ophthalmology)
-- Scenario: Forecast 24 months ahead to model LOE impact

-- Paso 1: Prepare clean training data
CREATE OR REPLACE VIEW eylea_training_data AS
SELECT 
    revenue_date,
    monthly_revenue_usd
FROM product_revenue_history
WHERE product_name = 'Eylea'
  AND revenue_date BETWEEN '2021-06-01' AND CURRENT_DATE()  -- Last 3 years
  AND monthly_revenue_usd IS NOT NULL                        -- No missing values
ORDER BY revenue_date;

-- Paso 2: Train model
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST eylea_forecast(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'eylea_training_data'),
    TIMESTAMP_COLNAME => 'revenue_date',
    TARGET_COLNAME => 'monthly_revenue_usd',
    CONFIG_OBJECT => {'lower_bound': 0}  -- Revenue can't be negative
);

-- Training complete in ~45 seconds!
```

### Why This Matters for Pharma

**Manual Forecasting** (old way):
- Analyst builds Excel models with trend lines
- Takes 2-4 hours per product
- Subjective assumptions ("looks like it's growing 5% per month")
- No confidence intervals
- Can't handle seasonality properly

**ML.FORECAST** (new way):
- Train model in 60 seconds per product
- Objective, data-driven predictions
- Automatic seasonality detection
- Confidence intervals for scenario planning
- Retrain weekly with new data

**Impacto de Negocio**: From **20 hours/month** (manual) to **30 minutes/month** (automated) for 10-product portfolio

### Technical Details

**Training Data Requirements**:
- **Minimum**: 10 data points (not recommended)
- **Recommended**: 
  - Monthly data: 24-36 points (2-3 years)
  - Quarterly data: 12-20 points (3-5 years)
  - Daily data: 100-365 points (3-12 months)
- **Maximum**: Unlimited (tested with 10+ years of data)

**Supported Algorithms** (auto-selected):
- **ARIMA**: Best for stationary series with clear autocorrelation
- **Prophet**: Best for strong seasonality + holidays
- **ETS**: Best for exponential smoothing scenarios

**Model Storage**:
- Stored as Snowflake object (like table or view)
- Includes trained weights, algorithm choice, preprocessing steps
- Secured with role-based access control

**Performance**:
- **Training**: 30-120 seconds per model
- **Forecasting**: <1 second for 24-month prediction
- **Parallel Training**: Train 100 product models in 2-3 minutes

### Common Patterns

**Multiple Products - Separate Models**:
```sql
-- Train model per product (recommended for different patterns)
CREATE SNOWFLAKE.ML.FORECAST product_a_forecast(INPUT_DATA => ...);
CREATE SNOWFLAKE.ML.FORECAST product_b_forecast(INPUT_DATA => ...);
```

**Portfolio View - Combined Training**:
```sql
-- Single model for entire portfolio (if similar patterns)
CREATE SNOWFLAKE.ML.FORECAST portfolio_forecast(
    INPUT_DATA => SYSTEM$QUERY_REFERENCE(
        'SELECT revenue_date, SUM(monthly_revenue) as total_revenue
         FROM product_revenue_history
         GROUP BY revenue_date'
    ),
    ...
);
```

**Incremental Retraining**:
```sql
-- Drop old model, create new one with updated data
DROP MODEL IF EXISTS xarelto_forecast;
CREATE SNOWFLAKE.ML.FORECAST xarelto_forecast(...);  -- Includes latest month
```

In [ ]:
-- Create views for each product's training data
CREATE OR REPLACE VIEW xarelto_training_data AS
SELECT revenue_date, monthly_revenue_usd 
FROM product_revenue_history 
WHERE product_name='Xarelto';

CREATE OR REPLACE VIEW eylea_training_data AS
SELECT revenue_date, monthly_revenue_usd 
FROM product_revenue_history 
WHERE product_name='Eylea';

-- Xarelto forecast model
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST xarelto_forecast(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'xarelto_training_data'),
    TIMESTAMP_COLNAME => 'revenue_date',
    TARGET_COLNAME => 'monthly_revenue_usd'
);

-- Eylea forecast model
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST eylea_forecast(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'eylea_training_data'),
    TIMESTAMP_COLNAME => 'revenue_date',
    TARGET_COLNAME => 'monthly_revenue_usd'
);

SELECT 'Forecast models trained!' as status;

---

## Paso 5: Generar 24-Month Forecasts

Use `!FORECAST()` to predict revenue with confidence intervals

### Calling Forecast Models

**Syntax**: `SELECT * FROM TABLE(model!FORECAST(FORECASTING_PERIODS => n))`

Returns columns:
- **`ts`**: Forecast timestamp
- **`forecast`**: Predicted value
- **`lower_bound`**: Pessimistic scenario (confidence interval)
- **`upper_bound`**: Optimistic scenario (confidence interval)

### Important: Table Functions

`!FORECAST()` is a **table function** that must be materialized:
- ✅ Use `CREATE TABLE AS SELECT` to store results
- ❌ Cannot use `CREATE VIEW` with table functions

In [ ]:
-- Generar forecasts for both products
CREATE OR REPLACE TABLE all_forecasts AS
SELECT 'Xarelto' as product, ts as forecast_date, forecast as predicted_revenue,
       lower_bound as pessimistic, upper_bound as optimistic
FROM TABLE(xarelto_forecast!FORECAST(FORECASTING_PERIODS => 24))
UNION ALL
SELECT 'Eylea', ts, forecast, lower_bound, upper_bound
FROM TABLE(eylea_forecast!FORECAST(FORECASTING_PERIODS => 24));

-- View forecasts
SELECT product, forecast_date, 
       ROUND(predicted_revenue/1000000,1) as pred_millions,
       ROUND(pessimistic/1000000,1) as pessimistic_millions,
       ROUND(optimistic/1000000,1) as optimistic_millions
FROM all_forecasts
WHERE MONTH(forecast_date) IN (1,6,12)
ORDER BY product, forecast_date LIMIT 15;

---

## Paso 6: LOE Risk Analysis

Calculate revenue at risk from upcoming patent expiries

In [ ]:
-- Portfolio risk summary
CREATE OR REPLACE VIEW loe_risk_analysis AS
WITH current_rev AS (
    SELECT product_name, AVG(monthly_revenue_usd) * 12 as current_annual
    FROM product_revenue_history
    WHERE revenue_date >= DATEADD('month', -3, CURRENT_DATE())
    GROUP BY product_name
),
forecast_rev AS (
    SELECT product, SUM(predicted_revenue) as forecast_annual
    FROM all_forecasts
    WHERE forecast_date >= CURRENT_DATE()
    AND forecast_date < DATEADD('year', 1, CURRENT_DATE())
    GROUP BY product
)
SELECT 
    le.product_name,
    le.patent_expiry_date,
    DATEDIFF('month', CURRENT_DATE(), le.patent_expiry_date) as months_to_loe,
    ROUND(cr.current_annual/1000000,0) as current_millions,
    ROUND(fr.forecast_annual/1000000,0) as forecast_millions,
    ROUND((fr.forecast_annual - cr.current_annual)/1000000,0) as revenue_change_millions,
    CASE 
        WHEN months_to_loe <= 12 THEN '🔴 High Risk'
        WHEN months_to_loe <= 24 THEN '🟡 Medium Risk'
        ELSE '🟢 Low Risk'
    END as risk_level
FROM loe_events le
JOIN current_rev cr ON le.product_name = cr.product_name
JOIN forecast_rev fr ON le.product_name = fr.product
ORDER BY months_to_loe;

SELECT * FROM loe_risk_analysis;

---

## Paso 7: Interactive Forecasting Dashboard

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("📉 LOE Revenue Forecaster")

# LOE risk summary
risk = session.sql("SELECT * FROM loe_risk_analysis ORDER BY months_to_loe").to_pandas()

col1, col2 = st.columns(2)
with col1:
    high_risk = len(risk[risk['RISK_LEVEL']=='🔴 High Risk'])
    st.metric("High Risk Products", high_risk)
with col2:
    total_risk = risk['REVENUE_CHANGE_MILLIONS'].sum()
    st.metric("Total Revenue at Risk", f"${total_risk:.0f}M")

st.subheader("⚠️ LOE Risk by Product")
st.dataframe(risk, use_container_width=True, hide_index=True)

# Forecast trends
st.subheader("📊 Revenue Forecasts")
product = st.selectbox("Select Product", ['Xarelto', 'Eylea'])

forecast = session.sql(f"""
    SELECT forecast_date, 
           ROUND(predicted_revenue/1000000,1) as forecast,
           ROUND(pessimistic/1000000,1) as pessimistic,
           ROUND(optimistic/1000000,1) as optimistic
    FROM all_forecasts WHERE product='{product}' ORDER BY forecast_date
""").to_pandas()

st.line_chart(forecast.set_index('FORECAST_DATE'))
st.caption("Revenue in millions - showing confidence intervals")

---

## ✅ Complete!

### What You Learned

1. ✅ `CREATE SNOWFLAKE.ML.FORECAST` for time-series
2. ✅ `!FORECAST(FORECASTING_PERIODS => 24)` generates predictions
3. ✅ Confidence intervals (upper/lower bounds) for scenarios
4. ✅ Portfolio risk analysis from LOE events

### Key Functions

**Training:**
```sql
CREATE SNOWFLAKE.ML.FORECAST model_name(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', '(SELECT ...)'),
    TIMESTAMP_COLNAME => 'date_col',
    TARGET_COLNAME => 'value_col'
);
```

**Forecasting:**
```sql
SELECT * FROM TABLE(model!FORECAST(FORECASTING_PERIODS => 24));
```

Returns: `ts`, `forecast`, `lower_bound`, `upper_bound`

### LOE Impact

- **Pre-LOE**: Normal growth (5-10% annually)
- **LOE Event**: Revenue cliff begins
- **Post-LOE**: 80-90% decline over 12-24 months

[Snowflake ML FORECAST Docs](https://docs.snowflake.com/en/user-guide/ml-functions/forecasting)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `LOE_FORECAST_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

⚠️ **Warning**: This action cannot be undone. All data and models will be permanently deleted.


In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS LOE_FORECAST_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;